# 11. Presets Gallery — 문서 꾸미기 프리셋

승승아빠 한글매크로를 참고하여 이식한 **11개 프리셋**. 각 프리셋은 한 줄
호출로 "보기 좋은 결과" 를 만듭니다.

실제 HWP 가 설치된 환경에서는 아래 코드를 직접 실행해 before/after 를
비교해 보세요.

## 목차

1. 줄무늬 표 (`striped_rows`)
2. 표 제목행/바닥행 (`table_header`, `table_footer`)
3. 문서 타이틀 박스 (`title_box`)
4. 소제목 바 (`subtitle_bar`)
5. 요약 박스 (`summary_box`)
6. 목차 자동 생성 (`toc`)
7. 쪽번호 (`page_numbers`)
8. 바탕쪽 테두리 (`page_border`)
9. 형광펜 토글 (`highlight_yellow`)


## Setup — 공통 준비

```python
from hwpapi.core import App
from pathlib import Path

app = App(is_visible=True)

# 테스트 문서 생성 헬퍼
def make_sample_table(rows=5, cols=4):
    """간단한 sales 데이터 표 삽입."""
    import random
    data = [["지역", "Q1", "Q2", "Q3"]]
    for region in ["서울", "부산", "대구", "인천"][:rows-1]:
        data.append([region] + [f"{random.randint(100, 999)}" for _ in range(cols-1)])
    app.insert_table(data=data)
```


## 1. 줄무늬 표 — `preset.striped_rows()`

가독성을 높이는 제브라 스트라이프. 헤더 행은 건너뜁니다.

```python
# 기본: 흰/연회색 교대
make_sample_table()
app.preset.striped_rows()

# 커스텀 색상 (3색 이상 순환 가능)
app.preset.striped_rows(colors=["#E8F4F8", "#FFFFFF"])

# 헤더 별도 색상
app.preset.striped_rows(
    colors=["#FFFFFF", "#F5F5F5"],
    header_color="#003366",        # 진파랑 헤더
)
```

**Before** → 밋밋한 표 / **After** → 줄별 구분 확실


![striped_rows() — Before / After](img/presets/striped_rows.png)

*striped_rows() — Before / After*


## 2. 표 제목행 / 바닥행 — `preset.table_header()`, `preset.table_footer()`

```python
make_sample_table()

# 상단 1행 — 하늘색 + 흰 글씨 굵게
app.preset.table_header(color="sky", text_color="#FFFFFF", bold=True)

# 하단 1행 (합계행 등) — 회색
app.preset.table_footer(color="gray", bold=True)

# 미리 정의된 색상 preset
# "gray" (#4D4D4D), "sky" (#9FC5E8), "dark_blue" (#003366),
# "green" (#6AA84F), "red" (#CC4125)
# 또는 hex 직접 지정
app.preset.table_header(color="#FF6600", rows=2)   # 2행 제목행
```


![table_header(color=...) — 기본 / sky / dark_blue](img/presets/table_header.png)

*table_header(color=...) — 기본 / sky / dark_blue*


## 3. 문서 타이틀 박스 — `preset.title_box()`

문서 상단에 1x2 박스 배치, 좌측 title / 우측 subtitle.

```python
# 기본
app.preset.title_box(
    text="2026년 상반기 업무 보고",
    subtitle="기획재무팀",
)

# 다크 테마
app.preset.title_box(
    text="회의록",
    subtitle="2026-04-15",
    bg_color="#003366",
    title_color="#FFFFFF",
    font_size=1400,
)

# 크기 조정
app.preset.title_box(
    text="제안서",
    width="180mm",
    height="30mm",
)
```


![title_box() — 기본 / 다크 테마](img/presets/title_box.png)

*title_box() — 기본 / 다크 테마*


## 4. 소제목 바 — `preset.subtitle_bar()`

한 줄 회색 배경 바에 소제목.

```python
app.preset.subtitle_bar("1. 개요")
app.insert_text("본 보고서는 ...")
app.insert_paragraph_break()

app.preset.subtitle_bar("2. 추진 경과", bg_color="#E0E0E0")
app.insert_text("...")
```


![subtitle_bar + summary_box + page_numbers 조합](img/presets/subtitle_summary.png)

*subtitle_bar + summary_box + page_numbers 조합*


## 5. 요약 박스 — `preset.summary_box()`

본문 강조용 박스.

```python
app.preset.summary_box(
    text="핵심 요약: Q3 매출 18% 증가, 신규 채널 3개 확보.",
    bg_color="#FFF9E0",    # 연노랑
)

# 단색 박스
app.preset.summary_box("중요!", bg_color="#FFE4E4")
```


## 6. 목차 자동 생성 — `preset.toc()`

개요 번호(제목1~N) 로 작성된 문단 기준으로 목차 + 책갈피 자동 생성.
PDF 변환 시 북마크 사이드바까지 생성.

```python
# 사전 작업: 각 섹션 제목을 app.insert_heading(level=N) 으로 표기
app.insert_heading("1장 도입", level=1)
app.insert_text("...")
app.insert_heading("1.1 배경", level=2)
app.insert_text("...")
app.insert_heading("2장 본론", level=1)
# ...

# 커서를 목차 위치로 이동 후
app.move.doc.top()
app.preset.toc(
    with_bookmarks=True,     # PDF 북마크 자동 생성
    dot_leader=True,         # 점끌기탭
    levels=3,                # 제목 1~3
)
```


![preset.toc() — 점끌기탭 + 페이지 번호](img/presets/toc.png)

*preset.toc() — 점끌기탭 + 페이지 번호*


## 7. 쪽번호 + 머리글 — `preset.page_numbers()`

```python
# 기본 — 하단 중앙 "1 / N"
app.preset.page_numbers()

# 하단 우측, "1" 형태만
app.preset.page_numbers(position="bottom-right", format="1")

# 머리글에 파일명 자동 삽입
app.preset.page_numbers(header_filename=True)

# 위치 옵션:
# "bottom-center" | "bottom-right" | "bottom-left"
# "top-center" | "top-right" | "top-left"
# 형식 옵션:
# "1 / N" | "1" | "-1-"
```


## 8. 바탕쪽 테두리 — `preset.page_border()`

편집 영역이 페이지 안에 잘 들어가는지 디버그할 때 유용.

```python
# 활성화 — 페이지 테두리 보임
app.preset.page_border(enable=True)

# 레이아웃 확인 후 제거
app.preset.page_border(enable=False)
```


## 9. 형광펜 토글 — `preset.highlight_yellow()`

`app.highlight(color)` 은 단방향 설정, `preset.highlight_yellow()` 은
**토글 방식** (노란색이면 해제, 아니면 노란색 적용).

```python
app.sel.current_word()
app.preset.highlight_yellow()    # 노란 배경 추가

# 다시 선택 후 호출하면 해제 (토글)
app.sel.current_word()
app.preset.highlight_yellow()    # 제거됨
```


![preset.highlight_yellow() — 선택 단어 노란 배경](img/presets/highlight_yellow.png)

*preset.highlight_yellow() — 선택 단어 노란 배경*


## 전체 문서 보기 좋게 만들기 — 조합 레시피

### 레시피 1 — 공공 보고서 스타일

```python
app = App()
app.new_document()

# 표지
app.preset.title_box(
    text="2026년 상반기 업무 보고서",
    subtitle="○○과",
    bg_color="#003366",
    title_color="#FFFFFF",
)
app.insert_paragraph_break()
app.insert_paragraph_break()

# 목차 placeholder
app.preset.subtitle_bar("목차")
# ... (내용 작성 후 마지막에 app.preset.toc())

# 본문 섹션들
for title in ["1. 개요", "2. 추진 경과", "3. 향후 계획"]:
    app.preset.subtitle_bar(title)
    app.insert_text("여기에 내용 ...")
    app.insert_paragraph_break()

# 쪽번호 + 머리글
app.preset.page_numbers(header_filename=True)

app.save("report.hwp")
```

### 레시피 2 — 컬러 보고서 (표 중심)

```python
app.insert_table(data=df)
app.preset.table_header(color="sky", text_color="#FFFFFF")
app.preset.striped_rows()
app.preset.table_footer(color="gray")   # 합계 행
```
